<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, glob, json
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
def load_dataset(dirs, min_blocks=5, n_feat=8):
    """
    dirs: a directory (or list) of *.json ACFG files.
    Groups functions by `fname` -> integer class label.
    Keeps only functions with >= min_blocks blocks.
    Selects the FIRST n_feat feature columns (8 = Gemini, 21 = all).
    Returns: funcs (list of dicts), labels (np.int64 array), classes (name->id).
    """

    if isinstance(dirs, str):
        dirs = [dirs]
    files = []
    for d in dirs:
        files += sorted(glob.glob(os.path.join(d, "*.json")))
        files += sorted(glob.glob(os.path.join(d, "*.jsonl")))
    if not files:
        raise FileNotFoundError(f"no .json under {dirs}")

    funcs, fnames, dropped = [], [], 0
    for fp in files:
        with open(fp) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                r = json.loads(line)
                feats = r["features"]
                if len(feats) < min_blocks:            # filter tiny functions
                    dropped += 1
                    continue

                # select first n_feat columns
                r["X"] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
                r["succs"] = r["succs"] if "succs" in r else None
                funcs.append(r)
                fnames.append(r["fname"])

    uniq = sorted(set(fnames))
    classes = {n: i for i, n in enumerate(uniq)}
    labels = np.array([classes[n] for n in fnames], dtype=np.int64)
    print(f"[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} "
          f"dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}")
    return funcs, labels, classes


In [35]:
funcs, labels, classes = load_dataset('/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg', min_blocks=0, n_feat=8)

[load] files=12 funcs=70968 classes=6627 dropped(<0 blocks)=0 n_feat=8


In [16]:
# labels[:10]
# classes['ASN1_PRINTABLE_new']

609

In [8]:
def make_or_load_split(labels, path="split.npz", ratios=(0.8, 0.1, 0.1), seed=0):
    """
    Split by CLASS identity (never by instance), so all compilations of one
    function land in the same bucket -> no identity leak. Deterministic.
    Returns three boolean masks (train, val, test) over the instances.
    """
    if os.path.exists(path):
        z = np.load(path)
        cls_perm = z["cls_perm"]
        print(f"[split] loaded {path} ({len(cls_perm)} classes)")
    else:
        n_cls = int(labels.max()) + 1
        rng = np.random.default_rng(seed)
        cls_perm = rng.permutation(n_cls)
        np.savez(path, cls_perm=cls_perm)
        print(f"[split] created {path} ({n_cls} classes, seed={seed})")

    n_cls = len(cls_perm)
    n_tr = int(ratios[0] * n_cls)
    n_va = int(ratios[1] * n_cls)
    train_cls = set(cls_perm[:n_tr].tolist())
    val_cls   = set(cls_perm[n_tr:n_tr + n_va].tolist())
    test_cls  = set(cls_perm[n_tr + n_va:].tolist())

    tr = np.array([c in train_cls for c in labels])
    va = np.array([c in val_cls   for c in labels])
    te = np.array([c in test_cls  for c in labels])
    print(f"[split] instances  train={tr.sum()}  val={va.sum()}  test={te.sum()}")
    return tr, va, te


In [26]:
def _norm(V, eps=1e-10):
    return V / (np.linalg.norm(V, axis=1, keepdims=True) + eps)


def _by_class(labels):
    d = {}
    for i, c in enumerate(labels):
        d.setdefault(int(c), []).append(i)
    return d


def auc_pairs(E, labels, n_pairs=8000, seed=0):
    """Balanced pos/neg pairs -> ROC-AUC and PR-AUC on cosine similarity."""
    rng = np.random.default_rng(seed)
    En = _norm(E)
    by = _by_class(labels)
    pos_c = [c for c, v in by.items() if len(v) >= 2]
    s, y = [], []
    for _ in range(n_pairs // 2):
        c = pos_c[rng.integers(len(pos_c))]
        i, j = rng.choice(by[c], 2, replace=False)
        s.append(float(En[i] @ En[j])); y.append(1)
    n = len(labels)
    got = 0
    while got < n_pairs // 2:
        i, j = rng.integers(n), rng.integers(n)
        if labels[i] != labels[j]:
            s.append(float(En[i] @ En[j])); y.append(0); got += 1
    return roc_auc_score(y, s), average_precision_score(y, s)


def retrieval(E, labels, pool_sizes=(10, 100, 1000), n_queries=500, ks=(1, 5), seed=0):
    """One-to-many search: put a query's twin in a pool of distractors, rank by cosine."""
    rng = np.random.default_rng(seed)
    En = _norm(E)
    by = _by_class(labels)
    pos_c = [c for c, v in by.items() if len(v) >= 2]
    out = {}
    for pool in pool_sizes:
        if pool > len(labels):
            continue
        rec = {k: 0 for k in ks}; mrr = 0.0; Q = 0
        for _ in range(n_queries):
            c = pos_c[rng.integers(len(pos_c))]
            q, tgt = rng.choice(by[c], 2, replace=False)
            # distractors: random instances NOT in the query's class
            distr = []
            while len(distr) < pool - 1:
                d = rng.integers(len(labels))
                if labels[d] != c:
                    distr.append(d)
            cand = np.array([tgt] + distr)
            sims = En[q] @ En[cand].T
            order = np.argsort(-sims)          # rank 0 = tgt
            rank = int(np.where(order == 0)[0][0]) + 1
            mrr += 1.0 / rank
            for k in ks:
                if rank <= k:
                    rec[k] += 1
            Q += 1
        for k in ks:
            out[f"recall@{k}_pool{pool}"] = rec[k] / Q
        out[f"mrr_pool{pool}"] = mrr / Q
    return out


In [27]:
def evaluate(embedder, funcs, labels, tag="", wandb_run=None,
             pool_sizes=(10, 100, 1000), n_pairs=8000, n_queries=500):
    E = embedder.embed_batch(funcs)
    auc, pr = auc_pairs(E, labels, n_pairs=n_pairs)
    ret = retrieval(E, labels, pool_sizes=pool_sizes, n_queries=n_queries)
    m = {"auc": auc, "pr_auc": pr, **ret}
    print(f"\n===== {tag} =====")
    print(f"pairwise ROC-AUC : {auc:.4f}   PR-AUC : {pr:.4f}")
    print(f"  pool |     R@1 |     R@5 |     MRR")
    print("  " + "-" * 34)
    for pool in pool_sizes:
        if f"recall@1_pool{pool}" in m:
            print(f"  {pool:4d} | {m[f'recall@1_pool{pool}']:.4f} | "
                  f"{m[f'recall@5_pool{pool}']:.4f} | {m[f'mrr_pool{pool}']:.4f}")
    if wandb_run is not None:
        wandb_run.log({f"{tag}/{k}": v for k, v in m.items()})
    return m


In [23]:
class BaselineEmbedder:
    """
    Collapse each function's N x F block-feature matrix into ONE vector by
    summarizing across blocks: mean, sum, max, std of each feature + block count.
    No training. Pure statistical fingerprint. Compared by cosine downstream.
    """
    def embed_batch(self, funcs):
        out = []
        for r in funcs:
            X = r["X"]
            agg = np.concatenate([
                X.mean(0), X.sum(0), X.max(0), X.std(0),
                [len(X)]                                 # block count
            ]).astype(np.float64)
            out.append(agg)
        E = np.asarray(out)
        # standardize columns so no single feature dominates the cosine
        mu, sd = E.mean(0), E.std(0) + 1e-8
        return (E - mu) / sd

In [32]:
if __name__ == "__main__":
    import sys
    DATA = '/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/zlib_acfg'
    funcs, labels, classes = load_dataset(DATA, min_blocks=5, n_feat=8)
    tr, va, te = make_or_load_split(labels, path="split.npz",  ratios=(0, 0, 1))

    test_funcs = [f for f, m in zip(funcs, te) if m]
    test_labels = labels[te]

    print("\n--- Rung 0: aggregate-feature baseline (test split) ---")
    # BaselineEmbedder().embed_batch(test_funcs)
    evaluate(BaselineEmbedder(), test_funcs, test_labels, tag="baseline")

[load] files=12 funcs=2955 classes=364 dropped(<5 blocks)=1079 n_feat=8
[split] loaded split.npz (4228 classes)
[split] instances  train=0  val=0  test=2955

--- Rung 0: aggregate-feature baseline (test split) ---

===== baseline =====
pairwise ROC-AUC : 0.9036   PR-AUC : 0.9083
  pool |     R@1 |     R@5 |     MRR
  ----------------------------------
    10 | 0.6480 | 0.9540 | 0.7739
   100 | 0.3560 | 0.6020 | 0.4767
  1000 | 0.1740 | 0.2760 | 0.2373


In [17]:
# [load] files=12 funcs=38058 classes=4228 dropped(<5 blocks)=32910 n_feat=8
# [split] loaded split.npz (4228 classes)
# [split] instances  train=30322  val=3900  test=3836

# --- Rung 0: aggregate-feature baseline (test split) ---

# ===== baseline =====
# pairwise ROC-AUC : 0.8552   PR-AUC : 0.8676
#   pool |     R@1 |     R@5 |     MRR
#   ----------------------------------
#     10 | 0.5580 | 0.9120 | 0.7029
#    100 | 0.3000 | 0.5240 | 0.4090
#   1000 | 0.1780 | 0.3040 | 0.2402